<a href="https://colab.research.google.com/github/JuniorProject21/DrugDiscovery/blob/main/DeepCoy_Attempt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dataset_clean.csv", sep=',')


In [ ]:
df_smiles = df['Ligand SMILES']


In [ ]:
import sys
sys.path.append("../")
sys.path.append("../evaluation/")
sys.path.append("/content/drive/MyDrive/")

In [ ]:
!pip install rdkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 41.8 MB/s eta 0:00:00


In [ ]:
import sys
import os
from google.colab import drive

# Ensure drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Fix for 'rdkit.six' module missing in newer RDKit versions
try:
    import rdkit.six
except ImportError:
    import six
    import sys
    # Create a shim for rdkit.six
    import types
    rdkit_six = types.ModuleType("rdkit.six")
    rdkit_six.iteritems = six.iteritems
    sys.modules["rdkit.six"] = rdkit_six
    print("Applied rdkit.six shim for compatibility.")

# Set path to DeepCoy source on Drive
drive_deepcoy = '/content/drive/MyDrive/'
sys.path.append(drive_deepcoy)
print(f'Added {drive_deepcoy} to system path.')

Applied rdkit.six shim for compatibility.
Added /content/drive/MyDrive/ to system path.


In [ ]:
!pip install rdkit docopt

  Preparing metadata (setup.py) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=aaf1a7eda8cdf4247303fdd822a45d95bb895890e953b180e719beae4bbcb14c
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [ ]:
%cd /content/drive/My\ Drive

/content/drive/My Drive


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.Draw import MolDrawing
from rdkit.Chem.Draw.rdMolDraw2D import MolDrawOptions as DrawingOptions
from rdkit.Chem import MolStandardize

import numpy as np
from itertools import product
from joblib import Parallel, delayed
import re
from collections import defaultdict
from IPython.display import clear_output
IPythonConsole.ipython_useSVG = True

import sys
import os
import importlib

# Force reload of utils to catch the new MLP definition
import utils
importlib.reload(utils)
print("Reloaded utils.py successfully.")

# These should now work with the updated utils.py
import DeepCoy
from DeepCoy import DenseGGNNChemModel
import prepare_data
from prepare_data import read_file, preprocess
import select_and_evaluate_decoys

Reloaded utils.py successfully.


In [ ]:
# Whether to use GPU for generating molecules with DeLinker
use_gpu = True

In [ ]:
newDataset = df[df['activity'] == 0]

newDataset = newDataset.drop(['BindingDB Reactant_set_id','IC50 (nM)','Target Source Organism According to Curator or DataSource','IC50_nM_clean','pIC50','activity'], axis = 1)
newDataset.reset_index(drop=True, inplace=True)
print(newDataset.head(20))



                                        Ligand SMILES
0      Clc1cccc(c1)N1CCN(CCCc2nc3ccccc3c(=O)[nH]2)CC1
1                   O=c1[nH]c2cc(CNCC#N)ccc2c2NCCCc12
2           CCCN1CCC(CC1)Oc1ccc2[nH]c(=O)c3CCCNc3c2c1
3       Cc1ccc2N(CCCc2c1)C(=O)CCc1nc2ccccc2c(=O)[nH]1
4   CC(C)(C)C1NCCc2cc(sc12)-c1cnn2c(cc(Cl)c2c1)C(N)=O
5                       Oc1cccc2c(O)nc(nc12)-c1ccccc1
6   Fc1cccc(c1)-c1nnc(o1)[C@H]1CC[C@@H](CC1)NC(=O)...
7         O=C(NCCN1CCCCC1)c1ccc2[nH]c(=O)c3cccnc3c2c1
8   COc1ccc(F)cc1\C=C\c1nnc(o1)-c1cc(C)cc2c1[nH]c(...
9   NC(=O)c1ccccc1OCC(=O)N1CC(=O)Nc2cc(ccc12)C(F)(F)F
10  COc1ccc(cc1)-c1nnc(o1)[C@H]1CC[C@@H](CC1)NC(=O...
11                    Nc1cccc2c1cc([nH]c2=O)-c1ccccc1
12            CN1CCC(CC1)Oc1ccc2[nH]c(=O)c3CCCNc3c2c1
13                 COCn1c2ccc(CCN(C)C)cc2c2NCCCc2c1=O
14                    CC(=O)CCCC(=O)Nc1cccc(c1)C(N)=O
15                 COc1ccc2c(CCN3C(=O)c4ccccc4C23O)c1
16  COc1ccc(cc1)C(=O)C1CCN(CC(=O)NCc2nc3CCCCc3c(=O...
17                    OCCOc1

In [ ]:
# ---- Define Xf and y ----
Xf = newDataset['Ligand SMILES'].values
y = np.zeros(len(newDataset))

# ---- Scaffold split ----
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
import numpy as np

def murcko(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

smiles_list = newDataset["Ligand SMILES"].astype(str).tolist()
scaffolds = [murcko(s) for s in smiles_list]

scaf_to_idx = defaultdict(list)
for i, sc in enumerate(scaffolds):
    if sc is not None:
        scaf_to_idx[sc].append(i)

groups = sorted(scaf_to_idx.values(), key=len, reverse=True)

test_fraction = 0.2
n_total = len(newDataset)
n_test_target = int(test_fraction * n_total)

test_idx = []
train_idx = []

for g in groups:
    if len(test_idx) < n_test_target:
        test_idx.extend(g)
    else:
        train_idx.extend(g)

train_idx = np.array(train_idx)
test_idx = np.array(test_idx)

X_train, X_test = Xf[train_idx], Xf[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (997,)
Test size: (255,)


[09:57:25] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 21
[09:57:25] Explicit valence for atom # 5 N, 4, is greater than permitted


In [ ]:
from rdkit import Chem

invalid_smiles = []
for i, smiles in enumerate(newDataset['Ligand SMILES']):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        invalid_smiles.append((i, smiles))

if invalid_smiles:
    print(f"Found {len(invalid_smiles)} invalid SMILES strings:")
    for idx, smiles_str in invalid_smiles:
        print(f"Index {idx}: {smiles_str}")
else:
    print("No invalid SMILES strings found in newDataset.")

Found 2 invalid SMILES strings:
Index 1157: Cc1cc(Cn2c3ccc(cc3c(=O)n(Cc3cnc(C)n3)c2=O)S(=O)(=O)NC2(C)CC2)n(C)n1
Index 1162: CC(C)OC[N]1=CN(Cc2cnn(C)c2)C(=O)c2cc(ccc12)S(=O)(=O)NC1(C)CC1


[09:57:28] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 21
[09:57:28] Explicit valence for atom # 5 N, 4, is greater than permitted


In [ ]:
invalid_indices = [idx for idx, _ in invalid_smiles]
newDataset = newDataset.drop(invalid_indices).reset_index(drop=True)

print(f"New dataset shape after dropping invalid SMILES: {newDataset.shape}")

New dataset shape after dropping invalid SMILES: (1252, 1)


In [ ]:
Xf = newDataset['Ligand SMILES'].values
y = np.zeros(len(newDataset))

print(f"Xf defined with shape: {Xf.shape}")
print(f"y defined with shape: {y.shape}")
print("First 5 SMILES in Xf:", Xf[:5])

Xf defined with shape: (1252,)
y defined with shape: (1252,)
First 5 SMILES in Xf: ['Clc1cccc(c1)N1CCN(CCCc2nc3ccccc3c(=O)[nH]2)CC1'
 'O=c1[nH]c2cc(CNCC#N)ccc2c2NCCCc12'
 'CCCN1CCC(CC1)Oc1ccc2[nH]c(=O)c3CCCNc3c2c1'
 'Cc1ccc2N(CCCc2c1)C(=O)CCc1nc2ccccc2c(=O)[nH]1'
 'CC(C)(C)C1NCCc2cc(sc12)-c1cnn2c(cc(Cl)c2c1)C(N)=O']


In [ ]:
import os
import sys
import tensorflow.compat.v1 as tf

# Enable TF1 compatibility globally
tf.disable_eager_execution()

# Force the sys.modules entry for 'tensorflow' to point to the v1 compat module
# This ensures any library (like DeepCoy) that runs 'import tensorflow'
# gets the v1 version with .ConfigProto
sys.modules['tensorflow'] = tf

print(f"TensorFlow 1.x compatibility forced globally. Module: {sys.modules['tensorflow']}")

if not use_gpu:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
else:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'

TensorFlow 1.x compatibility forced globally. Module: <module 'tensorflow.compat.v1' from '/usr/local/lib/python3.12/dist-packages/tensorflow/_api/v2/compat/v1/__init__.py'>


In [ ]:
# Arguments for DeepCoy
args = defaultdict(None)
args['--dataset'] = 'zinc'
args['--config'] = '{"generation": true, \
                     "batch_size": 1, \
                     "number_of_generation_per_valid": 100, \
                     "train_file": "molecules_P38-alpha_actives.json", \
                     "valid_file": "molecules_P38-alpha_actives.json", \
                     "output_name": "P38-alpha_example_decoys.smi", \
                     "use_subgraph_freqs": false}'
args['--freeze-graph-model'] = False
args['--restore'] = '../models/DeepCoy_DUDE_model_e09.pickle'

In [ ]:
import json
import os

# Define paths
drive_path = '/content/drive/MyDrive/'
deepcoy_data_dir = os.path.join(drive_path, 'deepcoy_data')
train_dir = os.path.join(deepcoy_data_dir, 'train')
valid_dir = os.path.join(deepcoy_data_dir, 'valid')
os.makedirs(train_dir, exist_ok=True)
os.makedirs(valid_dir, exist_ok=True)

# DeepCoy's preprocess function expects a specific structure
train_raw = [{'smiles_1': s, 'smiles_2': s} for s in X_train]
valid_raw = [{'smiles_1': s, 'smiles_2': s} for s in X_test]

# Run preprocess from DeepCoy
# This saves molecules_train.json and molecules_valid.json
prepare_data.preprocess(train_raw, dataset='zinc', name='train', save_dir=train_dir)
prepare_data.preprocess(valid_raw, dataset='zinc', name='valid', save_dir=valid_dir)

train_file_for_deepcoy = os.path.join(train_dir, 'molecules_train.json')
valid_file_for_deepcoy = os.path.join(valid_dir, 'molecules_valid.json')

print(f"Data prepared at: {valid_file_for_deepcoy}")

Parsing smiles as graphs.
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type S3(1)
unrecognized atom type S3(1)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
Processed: 997 / 997
Failed 4 molecules
Saving data.
Length raw data: 	997
Length processed data: 	993
Parsing smiles as graphs.
Processed: 255 / 255
Saving data.
Length raw data: 	255
Length processed data: 	255
Data prepared at: /content/drive/MyDrive/deepcoy_data/valid/molecules_valid.json


Since there is no "DeepCoy_DUDE_model_e09 file", we will be unable to train this model.

In [ ]:
import os
model_path = '/content/drive/MyDrive/models/DeepCoy_DUDE_model_e09.pickle'
if os.path.exists(model_path):
    size = os.path.getsize(model_path)
    print(f'File exists. Size: {size} bytes.')
    if size == 0:
        print('The file is empty and corrupted. Please delete it from Drive and upload a valid version.')
else:
    print('Model file not found. Please upload the .pickle file to the models folder.')

File exists. Size: 0 bytes.
The file is empty and corrupted. Please delete it from Drive and upload a valid version.


In [ ]:
import json
import os

# Define paths
train_file_path = '/content/drive/MyDrive/deepcoy_data/train/molecules_train.json'
valid_file_path = '/content/drive/MyDrive/deepcoy_data/valid/molecules_valid.json'
train_dir = os.path.dirname(train_file_path)
valid_dir = os.path.dirname(valid_file_path)

# Prepare raw data from X_train and X_test
train_raw_data = [{'smiles_1': s, 'smiles_2': s} for s in X_train]
valid_raw_data = [{'smiles_1': s, 'smiles_2': s} for s in X_test]

# Run the actual DeepCoy preprocessing to generate graphs
# This will overwrite the files with the 'graph_in' and 'graph_out' keys
print("Processing training set into graphs...")
prepare_data.preprocess(train_raw_data, dataset='zinc', name='train', save_dir=train_dir)

print("\nProcessing validation set into graphs...")
prepare_data.preprocess(valid_raw_data, dataset='zinc', name='valid', save_dir=valid_dir)

print("\nPreprocessing complete. Graph data saved to JSON files.")

Processing training set into graphs...
Parsing smiles as graphs.
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
unrecognized atom type S3(1)
unrecognized atom type S3(1)
unrecognized atom type P5(0)
unrecognized atom type P5(0)
Processed: 997 / 997
Failed 4 molecules
Saving data.
Length raw data: 	997
Length processed data: 	993

Processing validation set into graphs...
Parsing smiles as graphs.
Processed: 255 / 255
Saving data.
Length raw data: 	255
Length processed data: 	255

Preprocessing complete. Graph data saved to JSON files.


In [ ]:
import os

train_file_path = '/content/drive/MyDrive/deepcoy_data/train/molecules_train.json'

print(f"Displaying content of {train_file_path}:")

if os.path.exists(train_file_path):
    # Using !head to display the first 20 lines of the file
    !head -n 20 "{train_file_path}"
else:
    print(f"Error: File not found at {train_file_path}")

Displaying content of /content/drive/MyDrive/deepcoy_data/train/molecules_train.json:
[]

In [ ]:
# Updated Arguments for DeepCoy using correct graph-structured files
import os

# These are the files identified in deepcoy_data that actually contain the graph data
train_path = '/content/drive/MyDrive/deepcoy_data/trainmolecules_train.json'
valid_path = '/content/drive/MyDrive/deepcoy_data/validmolecules_valid.json'
model_path = '/content/drive/MyDrive/models/DeepCoy_DUDE_model_e09.pickle'
output_path = '/content/drive/MyDrive/deepcoy_data/newDataset_decoys.smi'

args = defaultdict(None)
args['--dataset'] = 'zinc'
args['--config'] = f'{{"generation": true, \
                     "batch_size": 1, \
                     "number_of_generation_per_valid": 10, \
                     "train_file": "{train_path}", \
                     "valid_file": "{valid_path}", \
                     "output_name": "{output_path}", \
                     "use_subgraph_freqs": false}}'
args['--freeze-graph-model'] = False
args['--restore'] = model_path

print("Configuration updated with correct graph file paths.")

Configuration updated with correct graph file paths.


In [ ]:
# Inspect the first few characters of the file to check for 'graph_in'
!head -c 1000 /content/drive/MyDrive/deepcoy_data/train/molecules_train.json

[]

using an older legacy version of keras

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import sys
import importlib
import types
import tensorflow.compat.v1 as tf_v1
tf_v1.disable_eager_execution()

# 1. Advanced Brute-force for Keras paths
import tf_keras
base_paths = ['keras', 'keras._tf_keras', 'keras._tf_keras.keras', 'tensorflow.python.keras']
for base in base_paths:
    sys.modules[base] = tf_keras
    sys.modules[f'{base}.__internal__'] = tf_keras.__internal__
    sys.modules[f'{base}.__internal__.legacy'] = tf_keras.__internal__
    sys.modules[f'{base}.__internal__.legacy.rnn_cell'] = tf_keras.__internal__

# 2. Custom DropoutWrapper to handle 'state_keep_prob'
class LegacyDropoutWrapper(tf_keras.layers.Dropout):
    def __init__(self, cell, state_keep_prob=1.0, **kwargs):
        rate = 1.0 - state_keep_prob
        super().__init__(rate=rate, **kwargs)
        self.cell = cell
    def __call__(self, inputs, state):
        return self.cell(inputs, state)

# 3. Patch RNN cells in the tf namespace directly
tf_v1.nn.rnn_cell.DropoutWrapper = LegacyDropoutWrapper
tf_v1.nn.rnn_cell.GRUCell = tf_keras.layers.GRUCell

# 4. Handle contrib.rnn manually
if not hasattr(tf_v1, 'contrib'):
    tf_v1.contrib = types.ModuleType("contrib")
    tf_v1.contrib.rnn = types.ModuleType("rnn")
    tf_v1.contrib.rnn.GRUCell = tf_keras.layers.GRUCell

# 5. Inject into DeepCoy
import DeepCoy
import GGNN_DeepCoy
importlib.reload(DeepCoy)
importlib.reload(GGNN_DeepCoy)
DeepCoy.tf = tf_v1
GGNN_DeepCoy.tf = tf_v1

print("Refined surgical patch complete. Ready for model initialization.")

# 6. Initialize and run
from DeepCoy import DenseGGNNChemModel
try:
    # Verify file exists before running
    model_path = args['--restore']
    if not os.path.exists(model_path):
        print(f"ERROR: Model weights not found at {model_path}. Please upload the .pickle file to Drive.")
    else:
        model = DenseGGNNChemModel(args)
        model.train()
        print(f"Decoy generation complete. Results saved to {args['--config']}")
except Exception as e:
    import traceback
    print(f"Initialization failed: {e}")
    traceback.print_exc()

Refined surgical patch complete. Ready for model initialization.
Run 2026-02-27-11-35-06_1734 starting with following parameters:
{"task_sample_ratios": {}, "use_edge_bias": true, "clamp_gradient_norm": 1.0, "out_layer_dropout_keep_prob": 1.0, "tie_fwd_bkwd": true, "random_seed": 0, "batch_size": 1, "num_epochs": 10, "epoch_to_generate": 10, "number_of_generation_per_valid": 10, "maximum_distance": 50, "use_argmax_generation": false, "residual_connection_on": true, "residual_connections": {"2": [0], "4": [0, 2], "6": [0, 2, 4], "8": [0, 2, 4, 6], "10": [0, 2, 4, 6, 8], "12": [0, 2, 4, 6, 8, 10], "14": [0, 2, 4, 6, 8, 10, 12]}, "num_timesteps": 7, "hidden_size": 100, "encoding_size": 8, "kl_trade_off_lambda": 0.3, "learning_rate": 0.001, "graph_state_dropout_keep_prob": 1, "compensate_num": 0, "train_file": "/content/drive/MyDrive/deepcoy_data/trainmolecules_train.json", "valid_file": "/content/drive/MyDrive/deepcoy_data/validmolecules_valid.json", "try_different_starting": true, "num_d

Instructions for updating:
keep_dims is deprecated, use keepdims instead


Restoring weights from file /content/drive/MyDrive/models/DeepCoy_DUDE_model_e09.pickle.
Initialization failed: Ran out of input


Traceback (most recent call last):
  File "/tmp/ipython-input-1734/1296711549.py", line 55, in <cell line: 0>
    model = DenseGGNNChemModel(args)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/DeepCoy.py", line 55, in __init__
    super().__init__(args)
  File "/content/drive/MyDrive/GGNN_DeepCoy.py", line 88, in __init__
    self.restore_model(restore_file)
  File "/content/drive/MyDrive/GGNN_DeepCoy.py", line 337, in restore_model
    data_to_load = pickle.load(in_file)
                   ^^^^^^^^^^^^^^^^^^^^
EOFError: Ran out of input


In [ ]:
import requests
# Checking the repository file list to find the correct path for models
repo_api_url = 'https://api.github.com/repos/oxpig/DeepCoy/contents/models'
response = requests.get(repo_api_url)
if response.status_code == 200:
    files = response.json()
    print('Files found in models directory:')
    for f in files:
        print(f['name'], f['download_url'])
else:
    print(f'Failed to access repository: {response.status_code}')

Failed to access repository: 404


trying to directly access the model through using the url provided in the github directory through the wayback machine


In [ ]:
import os
model_dir = '/content/drive/MyDrive/models/'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'DeepCoy_DUDE_model_e09.pickle')

# Remove the 0-byte corrupted file if it exists
if os.path.exists(model_path) and os.path.getsize(model_path) == 0:
    os.remove(model_path)
    print('Removed corrupted 0-byte file.')

# Attempting to download from a Wayback Machine snapshot of the OPIG resource
wayback_url = 'https://web.archive.org/web/20230524141505/http://opig.stats.ox.ac.uk/resources/DeepCoy/DeepCoy_DUDE_model_e09.pickle'

print(f'Attempting download from Wayback Machine: {wayback_url}')
!wget -O "{model_path}" "{wayback_url}"

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000:
    print(f'Success! Downloaded {os.path.getsize(model_path)} bytes to {model_path}')
else:
    print('Wayback download failed or file is too small. The file might not have been archived.')

## Search for Alternative Mirrors

### Subtask:
Search for alternative hosting sites and mirrors for the DeepCoy pretrained model weights (`DeepCoy_DUDE_model_e09.pickle`).


In [ ]:
import requests
import time

def search_deepcoy_mirrors():
    # 1. Search for forks of the oxpig/DeepCoy repo
    print('Searching for GitHub forks of oxpig/DeepCoy...')
    search_url = 'https://api.github.com/search/repositories?q=DeepCoy+fork:true'
    try:
        response = requests.get(search_url)
        if response.status_code != 200:
            print(f'GitHub Search API failed: {response.status_code}')
            return

        items = response.json().get('items', [])
        print(f'Found {len(items)} repositories matching "DeepCoy". Checking for model files...')

        valid_urls = []
        for repo in items:
            full_name = repo['full_name']
            # Check if models/DeepCoy_DUDE_model_e09.pickle exists in this fork
            check_url = f'https://api.github.com/repos/{full_name}/contents/models/DeepCoy_DUDE_model_e09.pickle'
            file_res = requests.get(check_url)

            if file_res.status_code == 200:
                download_url = file_res.json().get('download_url')
                if download_url:
                    print(f'[FOUND] Model found in fork: {full_name}')
                    valid_urls.append(download_url)

            # Rate limiting safety
            time.sleep(1)

        # 2. Check Zenodo/Figshare as a fallback
        print('\nChecking Zenodo API for "DeepCoy"...')
        zenodo_url = 'https://zenodo.org/api/records/?q=DeepCoy'
        z_res = requests.get(zenodo_url)
        if z_res.status_code == 200:
            hits = z_res.json().get('hits', {}).get('hits', [])
            for hit in hits:
                print(f'[INFO] Potential Zenodo archive: {hit["links"]["html"]}')

        if valid_urls:
            print('\nValid download URLs found:')
            for url in valid_urls:
                print(url)
        else:
            print('\nNo direct .pickle mirrors found on GitHub forks.')

    except Exception as e:
        print(f'An error occurred: {e}')

search_deepcoy_mirrors()

In [ ]:
import requests
import time

def search_deepcoy_mirrors_v2():
    print('Searching for GitHub forks of oxpig/DeepCoy...')
    search_url = 'https://api.github.com/search/repositories?q=DeepCoy+fork:true'
    valid_urls = []
    try:
        response = requests.get(search_url)
        if response.status_code == 200:
            items = response.json().get('items', [])
            print(f'Found {len(items)} repositories. Checking for model files...')
            for repo in items[:10]: # Check top 10 for speed
                full_name = repo['full_name']
                check_url = f'https://api.github.com/repos/{full_name}/contents/models/DeepCoy_DUDE_model_e09.pickle'
                file_res = requests.get(check_url)
                if file_res.status_code == 200:
                    d_url = file_res.json().get('download_url')
                    if d_url:
                        print(f'[FOUND GitHub] {full_name}')
                        valid_urls.append(d_url)
                time.sleep(0.5)
        else:
            print(f'GitHub API error: {response.status_code}')

        print('\nChecking Zenodo API...')
        zenodo_url = 'https://zenodo.org/api/records/?q=DeepCoy'
        z_res = requests.get(zenodo_url)
        if z_res.status_code == 200:
            hits = z_res.json().get('hits', {}).get('hits', [])
            for hit in hits:
                title = hit.get('metadata', {}).get('title', 'No Title')
                # Zenodo links are often in hit['links']['latest_html'] or hit['links']['self']
                link = hit.get('links', {}).get('self', 'No Link')
                print(f'[INFO Zenodo] Found: {title} - {link}')

        if valid_urls:
            print('\nSummary of valid download URLs:')
            for url in valid_urls:
                print(url)
        else:
            print('\nNo direct .pickle mirrors found yet.')
    except Exception as e:
        print(f'Error during search: {e}')

search_deepcoy_mirrors_v2()

In [ ]:
import requests

# Inspect the Zenodo record 7943200 found in the previous step
zenodo_api_record = 'https://zenodo.org/api/records/7943200'
response = requests.get(zenodo_api_record)

if response.status_code == 200:
    data = response.json()
    print(f"Title: {data.get('metadata', {}).get('title')}")
    files = data.get('files', [])
    if files:
        print('\nFiles in this record:')
        for f in files:
            print(f"- {f['key']} ({f['size']} bytes) -> {f['links']['self']}")
    else:
        print('\nNo files found directly in this record metadata.')
else:
    print(f'Failed to retrieve Zenodo record: {response.status_code}')

In [ ]:
import os
import tarfile
import requests

dataset_url = 'https://zenodo.org/api/records/7943200/files/zenodo_datasets.tar.gz/content'
tar_path = '/content/zenodo_datasets.tar.gz'
extract_path = '/content/zenodo_extracted'

print('Downloading zenodo_datasets.tar.gz (this may take a few minutes)...')
response = requests.get(dataset_url, stream=True)
if response.status_code == 200:
    with open(tar_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)
    print('Download complete.')

    print('Listing contents of the archive...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        # List only the first 50 members to get an idea of the structure
        members = tar.getmembers()
        print(f'Total files in archive: {len(members)}')
        print('Sample of files:')
        for m in members[:50]:
            print(m.name)
            if 'pickle' in m.name.lower():
                print(f'[FOUND POTENTIAL MODEL] {m.name}')
else:
    print(f'Failed to download archive: {response.status_code}')

In [ ]:
import tarfile

tar_path = '/content/zenodo_datasets.tar.gz'

print('Searching for .pickle files in the entire archive...')
with tarfile.open(tar_path, 'r:gz') as tar:
    all_members = tar.getnames()
    pickle_files = [name for name in all_members if name.endswith('.pickle')]

    if pickle_files:
        print(f'Found {len(pickle_files)} pickle files:')
        for f in pickle_files:
            print(f)
    else:
        print('No .pickle files found in the archive.')

# Also checking for files that might contain 'model' in their name but different extension
model_related = [name for name in all_members if 'model' in name.lower()]
if model_related:
    print(f'\nFound {len(model_related)} other model-related files/directories:')
    for f in model_related[:20]: # Limit output
        print(f)

In [ ]:
import requests
import time

def exhaustive_github_search():
    # Searching for any repository containing the string 'DeepCoy_DUDE_model' in its code/files
    print('Performing exhaustive GitHub code search for the model filename...')
    search_url = 'https://api.github.com/search/code?q=DeepCoy_DUDE_model+extension:pickle'
    headers = {'Accept': 'application/vnd.github.v3+json'}

    try:
        response = requests.get(search_url, headers=headers)
        if response.status_code == 200:
            items = response.json().get('items', [])
            if items:
                print(f'Found {len(items)} matching files in GitHub code search:')
                for item in items:
                    repo_name = item['repository']['full_name']
                    file_path = item['path']
                    download_url = item.get('html_url', '').replace('blob', 'raw')
                    print(f'[FOUND] {repo_name} -> {file_path}')
                    print(f'Raw URL: {download_url}')
            else:
                print('No files found in GitHub code search.')
        else:
            print(f'GitHub Code Search API failed: {response.status_code}. You may need an OAUTH token for code search.')

        # Fallback: Search for all pickles in the most likely forks found earlier
        print('\nChecking most promising forks for any pickle files in any directory...')
        top_forks = ['gmandic/DeepCoy', 'clinfo/DeepCoy', 'shiwentao00/DeepCoy']
        for fork in top_forks:
            tree_url = f'https://api.github.com/repos/{fork}/git/trees/master?recursive=1'
            tree_res = requests.get(tree_url)
            if tree_res.status_code == 200:
                tree = tree_res.json().get('tree', [])
                pickles = [node['path'] for node in tree if node['path'].endswith('.pickle')]
                if pickles:
                    print(f'Found pickles in {fork}: {pickles}')
            time.sleep(1)

    except Exception as e:
        print(f'Error during exhaustive search: {e}')

exhaustive_github_search()

In [ ]:
import os

def scan_for_pickles(start_path):
    print(f'Scanning {start_path} for any pickle files...')
    found_pickles = []
    for root, dirs, files in os.walk(start_path):
        for file in files:
            if file.endswith('.pickle'):
                full_path = os.path.join(root, file)
                size = os.path.getsize(full_path)
                found_pickles.append((full_path, size))
    return found_pickles

# Scan current working directory and Drive
local_pickles = scan_for_pickles('/content/')
drive_pickles = scan_for_pickles('/content/drive/MyDrive/') if os.path.exists('/content/drive/MyDrive/') else []

all_found = local_pickles + drive_pickles

if all_found:
    print(f'\nFound {len(all_found)} pickle files:')
    for path, size in all_found:
        print(f'- {path} ({size} bytes)')
else:
    print('\nNo pickle files found in accessible directories.')

In [ ]:
import json
import os
from collections import defaultdict

# 1. Verify JSON file contents
train_json_path = '/content/drive/MyDrive/deepcoy_data/train/molecules_train.json'
valid_json_path = '/content/drive/MyDrive/deepcoy_data/valid/molecules_valid.json'

def verify_json(path, name):
    if os.path.exists(path):
        with open(path, 'r') as f:
            data = json.load(f)
            print(f'{name} file loaded. Total entries: {len(data)}')
            if len(data) > 0:
                print(f'Sample entry keys from {name}: {list(data[0].keys())}')
                return data
    else:
        print(f'ERROR: {name} file not found at {path}')
    return None

train_data = verify_json(train_json_path, 'Training')
valid_data = verify_json(valid_json_path, 'Validation')

# 2. Update args for a 'tiny' training run
# We define a new path for the placeholder model to avoid EOF errors
placeholder_model_path = '/content/drive/MyDrive/models/DeepCoy_placeholder_model.pickle'
if os.path.exists(placeholder_model_path):
    os.remove(placeholder_model_path)

args = defaultdict(None)
args['--dataset'] = 'zinc'
# generation: false for training mode
args['--config'] = json.dumps({
    "generation": false,
    "batch_size": 10,
    "num_epochs": 1,
    "train_file": train_json_path,
    "valid_file": valid_json_path,
    "output_name": "placeholder_output.smi",
    "use_subgraph_freqs": false
})
args['--freeze-graph-model'] = False
args['--restore'] = placeholder_model_path # Set to non-existent path for fresh init

print('\nConfiguration updated for placeholder training.')
print(f'Model will attempt to save to: {placeholder_model_path}')

## Search Zenodo and Figshare



In [ ]:
import requests
import json

def search_zenodo(query):
    print(f'--- Searching Zenodo for "{query}" ---')
    url = f'https://zenodo.org/api/records/?q={query}'
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            hits = response.json().get('hits', {}).get('hits', [])
            if not hits:
                print('No Zenodo records found.')
            for hit in hits:
                title = hit.get('metadata', {}).get('title')
                record_id = hit.get('id')
                print(f'\nRecord: {title} (ID: {record_id})')
                files = hit.get('files', [])
                for f in files:
                    key = f.get('key', 'unknown')
                    link = f.get('links', {}).get('self')
                    if any(ext in key.lower() for ext in ['.pickle', '.zip', '.tar.gz']):
                        print(f'  [MATCH] {key} -> {link}')
        else:
            print(f'Zenodo API error: {response.status_code}')
    except Exception as e:
        print(f'Zenodo search failed: {e}')

def search_figshare(query):
    print(f'\n--- Searching Figshare for "{query}" ---')
    url = 'https://api.figshare.com/v2/articles/search'
    data = {'search_for': query, 'page_size': 10}
    try:
        response = requests.post(url, json=data, timeout=15)
        if response.status_code == 200:
            articles = response.json()
            if not articles:
                print('No Figshare articles found.')
            for article in articles:
                title = article.get('title')
                article_url = article.get('url_public_html')
                print(f'\nArticle: {title}')
                print(f'  Link: {article_url}')
        else:
            print(f'Figshare API error: {response.status_code}')
    except Exception as e:
        print(f'Figshare search failed: {e}')

# Execute searches
search_term = 'DeepCoy'
search_zenodo(search_term)
search_figshare(search_term)

## Fix Placeholder Training Script

### Subtask:
Update the model configuration to fix a syntax error and prepare for a potential placeholder training run.


**Reasoning**:
I will update the model configuration dictionary to fix the syntax error (Python False instead of lowercase false), ensure absolute paths for the graph data are correct, and set a new restore path for the placeholder model initialization.



In [ ]:
import json
import os
from collections import defaultdict

# 1. Define absolute paths based on previous directory inspection
train_json_path = '/content/drive/MyDrive/deepcoy_data/train/molecules_train.json'
valid_json_path = '/content/drive/MyDrive/deepcoy_data/valid/molecules_valid.json'
placeholder_model_path = '/content/drive/MyDrive/models/DeepCoy_placeholder_model.pickle'

# Ensure directory exists for the placeholder model
os.makedirs(os.path.dirname(placeholder_model_path), exist_ok=True)

# 2. Fix the NameError by using Python False and correct JSON formatting
args = defaultdict(None)
args['--dataset'] = 'zinc'

# Configuration dictionary using Python booleans
config_dict = {
    "generation": False,
    "batch_size": 10,
    "num_epochs": 1,
    "train_file": train_json_path,
    "valid_file": valid_json_path,
    "output_name": "placeholder_output.smi",
    "use_subgraph_freqs": False
}

# Convert to JSON string for the model
args['--config'] = json.dumps(config_dict)
args['--freeze-graph-model'] = False
args['--restore'] = placeholder_model_path

print("Configuration args updated successfully.")
print(f"Train file path: {train_json_path}")
print(f"Restore path (fresh init): {placeholder_model_path}")

## Verify Graph Data in JSON

### Subtask:
Investigate why the training and validation files loaded with 0 entries and update the configuration to point to the correct graph-structured data files.


**Reasoning**:
I will check the file existence and sizes for both the original and alternative paths to identify which files contain the actual graph data, then verify their content by loading them.



In [ ]:
import os
import json

# Paths to check
paths_to_verify = {
    'Original Train': '/content/drive/MyDrive/deepcoy_data/train/molecules_train.json',
    'Original Valid': '/content/drive/MyDrive/deepcoy_data/valid/molecules_valid.json',
    'Alt Train': '/content/drive/MyDrive/deepcoy_data/trainmolecules_train.json',
    'Alt Valid': '/content/drive/MyDrive/deepcoy_data/validmolecules_valid.json'
}

# 1 & 2. Check existence and sizes
found_data_paths = {}
for label, path in paths_to_verify.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{label}: Exists={exists}, Size={size} bytes")
    if exists and size > 0:
        found_data_paths[label] = path

# 3. Load and verify entries
for label, path in found_data_paths.items():
    try:
        with open(path, 'r') as f:
            data = json.load(f)
            print(f"SUCCESS: Loaded {label} from {path}. Entries: {len(data)}")
    except Exception as e:
        print(f"ERROR loading {label}: {e}")

# 4 & 5. Update configuration if alternative paths are the correct ones
if 'Alt Train' in found_data_paths and 'Alt Valid' in found_data_paths:
    train_json_path = paths_to_verify['Alt Train']
    valid_json_path = paths_to_verify['Alt Valid']

    config_dict['train_file'] = train_json_path
    config_dict['valid_file'] = valid_json_path
    args['--config'] = json.dumps(config_dict)
    print("\nUpdated config_dict and args with Alt paths.")
else:
    print("\nCould not automatically find valid Alt paths for both train and valid sets.")

## Final Task

### Subtask:
Provide a summary of the available model sources and, if still missing, initiate a minimal training run to ensure the pipeline structure is valid.


In [ ]:
import os
import requests
import tarfile

dataset_url = 'https://zenodo.org/api/records/7943200/files/zenodo_datasets.tar.gz/content'
tar_path = '/content/zenodo_datasets.tar.gz'
extract_path = '/content/zenodo_extracted'
model_dest_dir = '/content/drive/MyDrive/models/'

# 1. Download
if not os.path.exists(tar_path):
    print('Downloading zenodo_datasets.tar.gz (1.1 GB)...')
    response = requests.get(dataset_url, stream=True)
    if response.status_code == 200:
        with open(tar_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)
        print('Download complete.')
    else:
        print(f'Failed to download: {response.status_code}')
else:
    print('Archive already exists.')

# 2. Extract and Search for Pickles
print('Extracting and searching for model files...')
os.makedirs(extract_path, exist_ok=True)
found_models = []

with tarfile.open(tar_path, 'r:gz') as tar:
    # We extract member by member to avoid filling disk if not needed
    for member in tar.getmembers():
        if member.name.endswith('.pickle'):
            tar.extract(member, path=extract_path)
            src = os.path.join(extract_path, member.name)
            dest = os.path.join(model_dest_dir, os.path.basename(member.name))
            os.makedirs(model_dest_dir, exist_ok=True)
            # Copy to Drive
            import shutil
            shutil.copy(src, dest)
            found_models.append(dest)
            print(f'[SUCCESS] Found and moved: {dest}')

if not found_models:
    print('No .pickle files were found in the Zenodo archive.')
else:
    print(f'\nTotal models recovered: {len(found_models)}')
    # Update args to use the first found model if applicable
    args['--restore'] = found_models[0]